# 10 — Medical Tokenizer & Pretraining Data

This notebook builds the pretraining corpus and trains a **medical-domain BPE tokenizer**.

**Pretraining corpus (two sources combined):**
1. `epfl-llm/guidelines` — 33k clinical guidelines (WHO, CDC, NICE)
2. `pubmed_qa` pqa_unlabeled — 60k biomedical abstracts with context paragraphs

**Pipeline:**
1. Load both datasets and extract raw text
2. Split into train / val / test and **save to `.txt` files**
3. Train a new BPE tokenizer on the combined medical training text
4. Compare it to the TinyStories tokenizer on benchmark medical terms
5. Tokenize all splits into binary `.npy` files for training

**Why two sources?**  
Guidelines give long-form structured clinical prose (~7k words/doc).  
PubMedQA gives concise biomedical abstract language (~300 words/doc).  
Together they cover a wider range of medical writing styles.

## 1 — Setup

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────
import os, sys

# ── Mount Drive and clone repo (Colab only) ───────────────────────────────
if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = "/content/slm-learning"
    if not os.path.exists(REPO_DIR):
        # Replace <your-token> with a GitHub PAT if repo is private,
        # or remove the token section if repo is public.
        import subprocess
        subprocess.run([
            "pip", "install", "-q",
            "tokenizers", "datasets", "tqdm", "matplotlib", "huggingface_hub"
        ], check=True)
        token = ""   # paste your GitHub PAT here if repo is private
        url   = f"https://{token}@github.com/Aman2394/slm-from-scratch.git" if token \
                else "https://github.com/Aman2394/slm-from-scratch.git"
        subprocess.run(["git", "clone", url, REPO_DIR], check=True)
else:
    # Local: notebook lives in notebooks/, repo root is one level up
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

# Add repo root to path so `import config`, `from model.gpt import GPT` etc. work
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [17]:
# !pip install datasets tokenizers tqdm
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.10.0
CUDA available: False


## 2 — Load & Extract Text

We load both datasets fully into memory (no streaming — each fits comfortably under 1 GB).

**Text extraction:**
- Guidelines: `clean_text` field (full guideline markdown, ~7k words/doc)
- PubMedQA unlabeled: `question` + `context["contexts"]` paragraphs + `long_answer`

**Expected time:** ~2–5 min (download + processing).

In [20]:
cfg.MED_PUBMEDQA_DATASET,cfg.MED_PUBMEDQA_PRETRAIN_SUBSET

('pubmed_qa', 'pqa_unlabeled')

In [21]:
from datasets import load_dataset
from tqdm import tqdm
import config as cfg

# ── Load Dataset 1: epfl-llm/guidelines ──────────────────────────────────────
print("Loading epfl-llm/guidelines...")
guidelines = load_dataset(cfg.MED_TEXTBOOK_DATASET, split="train")
print(f"  {len(guidelines):,} examples")

# ── Load Dataset 2: PubMedQA unlabeled ───────────────────────────────────────
print("Loading pubmed_qa pqa_unlabeled...")
pubmedqa_ul = load_dataset(
    cfg.MED_PUBMEDQA_DATASET,
    cfg.MED_PUBMEDQA_PRETRAIN_SUBSET,
    split="train",
)
print(f"  {len(pubmedqa_ul):,} examples")

Loading epfl-llm/guidelines...
  37,970 examples
Loading pubmed_qa pqa_unlabeled...
  61,249 examples


In [22]:
def extract_guidelines_text(ex):
    """Return the clean text of a guidelines document."""
    return ex["clean_text"].strip()


def extract_pubmedqa_text(ex):
    """Concatenate question + context paragraphs + long_answer into one document."""
    parts = [ex["question"]] + ex["context"]["contexts"]
    if ex.get("long_answer"):
        parts.append(ex["long_answer"])
    return " ".join(parts).strip()


# ── Extract all texts ────────────────────────────────────────────────────────
guidelines_texts = [
    extract_guidelines_text(ex)
    for ex in tqdm(guidelines, desc="Guidelines")
]

pubmedqa_texts = [
    extract_pubmedqa_text(pubmedqa_ul[i])
    for i in tqdm(
        range(min(cfg.MED_PUBMEDQA_PRETRAIN_SIZE, len(pubmedqa_ul))),
        desc="PubMedQA",
    )
]

print(f"\nExtracted:")
print(f"  {len(guidelines_texts):,} guidelines documents")
print(f"  {len(pubmedqa_texts):,} pubmedqa documents")

PubMedQA: 100%|██████████| 60000/60000 [00:02<00:00, 20414.81it/s]



Extracted:
  37,970 guidelines documents
  60,000 pubmedqa documents


In [24]:
import textwrap
print(textwrap.fill(random.choice(pubmedqa_texts)))

Minor surgery in primary care: is continuing education within the team
a valid strategy for improving quality? To evaluate the impact of
continuing education within the team (FCI, in Spanish) on the quality
of minor surgery. Study of level of quality. SETTING. Primary care.
First evaluation: all the lesions referred for biopsy during 1998 (62
samples). Second evaluation: those referred in 1999-2000 (150). Four
explicit criteria regulating procedure and result were designed: C1,
sufficient information; C2, correct referral; C3, correct extirpation
of lesion; C4, clinical-pathological concordance. Request forms and
anatomical-pathological reports were assessed. Evaluation was before
and after corrective measures (FCI and organisational changes designed
to support FCI). The Kappa index of inter-observer concordance, the
Compliance Index and Fisher's Z index were analysed. 62 lesions were
included in the first evaluation, with high reliability for C1 and C4,
good for C2 and moderate for C3

## 3 — Split and Save to Disk

We split each source into train / val / test before writing to `.txt` files.
Val and test each get 1,000 docs from guidelines + 1,000 from PubMedQA so that
validation loss reflects the **full training distribution**, not just one source.

Each document is written as one line (newlines inside the document are collapsed
to spaces). The tokenizer and `tokenize_and_save()` both process one line at a time.

**Split sizes** (from `config.py`):
- Train: 33k guidelines + 58k PubMedQA ≈ 91k documents
- Val / Test: 1,000 guidelines + 1,000 PubMedQA = 2,000 each

In [25]:
import random
import config as cfg

# ── Per-source val/test sizes ─────────────────────────────────────────────────
VAL_PER_SOURCE  = cfg.MED_VAL_SIZE  // 2   # 1,000 each source
TEST_PER_SOURCE = cfg.MED_TEST_SIZE // 2   # 1,000 each source

# Guidelines: first 33k for train, next 1k val, next 1k test
gl_train = guidelines_texts[: cfg.MED_TEXTBOOK_DATA_SIZE]
gl_val   = guidelines_texts[cfg.MED_TEXTBOOK_DATA_SIZE : cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE]
gl_test  = guidelines_texts[cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE : cfg.MED_TEXTBOOK_DATA_SIZE + VAL_PER_SOURCE + TEST_PER_SOURCE]

# PubMedQA: last 2k reserved for val+test, the rest is train
pm_train = pubmedqa_texts[: -(VAL_PER_SOURCE + TEST_PER_SOURCE)]
pm_val   = pubmedqa_texts[-(VAL_PER_SOURCE + TEST_PER_SOURCE) : -TEST_PER_SOURCE]
pm_test  = pubmedqa_texts[-TEST_PER_SOURCE :]

# Combine and shuffle each split
train_texts = gl_train + pm_train
val_texts   = gl_val   + pm_val
test_texts  = gl_test  + pm_test

random.shuffle(train_texts)
random.shuffle(val_texts)
random.shuffle(test_texts)

print(f"Split sizes:")
print(f"  train : {len(train_texts):,}  ({len(gl_train):,} guidelines + {len(pm_train):,} pubmedqa)")
print(f"  val   : {len(val_texts):,}   ({len(gl_val):,} guidelines + {len(pm_val):,} pubmedqa)")
print(f"  test  : {len(test_texts):,}   ({len(gl_test):,} guidelines + {len(pm_test):,} pubmedqa)")

Split sizes:
  train : 91,000  (33,000 guidelines + 58,000 pubmedqa)
  val   : 2,000   (1,000 guidelines + 1,000 pubmedqa)
  test  : 2,000   (1,000 guidelines + 1,000 pubmedqa)


In [26]:
import os


def save_split(texts, out_path):
    """
    Write a list of document strings to a plain-text file, one document per line.

    Newlines inside each document are collapsed to spaces so that the file
    format stays line-per-document — required by ByteLevelBPETokenizer.train()
    and by tokenize_and_save() which iterates line by line.

    Args:
        texts    : list of document strings
        out_path : destination file path (created if missing)
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for doc in texts:
            # Collapse internal newlines so each document is exactly one line
            line = doc.replace("\n", " ").replace("\r", " ").strip()
            if line:
                f.write(line + "\n")
    print(f"  Saved {len(texts):,} docs → {out_path}")


print("Saving text splits to disk...")
save_split(train_texts, cfg.MED_TRAIN_TXT)
save_split(val_texts,   cfg.MED_VAL_TXT)
save_split(test_texts,  cfg.MED_TEST_TXT)
print("\nAll splits saved.")

Saving text splits to disk...
  Saved 91,000 docs → /Users/aman2394/Desktop/slm-learning/data/medical/train.txt
  Saved 2,000 docs → /Users/aman2394/Desktop/slm-learning/data/medical/val.txt
  Saved 2,000 docs → /Users/aman2394/Desktop/slm-learning/data/medical/test.txt

All splits saved.


## 4 — Train Medical BPE Tokenizer

We train a **brand-new** BPE tokenizer from scratch on `MED_TRAIN_TXT`.
This file already contains text from both sources (guidelines + PubMedQA),
so the tokenizer learns merges for the full pretraining vocabulary.

**Why retrain from scratch instead of reusing the TinyStories tokenizer?**  
Medical text contains very different frequent subword patterns — clinical
abbreviations, Latin roots, drug suffixes (-olol, -pril, -mab). A tokenizer
trained on children's stories will split these into many short fragments.
A medical-specific tokenizer merges them into single tokens, reducing sequence
length and making it easier for the model to learn domain patterns.

We keep `cfg.VOCAB_SIZE` (8192) to stay architecture-compatible.

**Expected time:** ~5 minutes.

In [28]:
from tokenizer.preprocess import train_tokenizer
import config as cfg

print(f"Training BPE tokenizer on : {cfg.MED_TRAIN_TXT}")
print(f"Target vocab size          : {cfg.VOCAB_SIZE:,}")
print(f"Output directory           : {cfg.MED_TOKENIZER_DIR}")

med_tok = train_tokenizer(
    train_file=cfg.MED_TRAIN_TXT,   # combined guidelines + pubmedqa text
    save_dir=cfg.MED_TOKENIZER_DIR,
    vocab_size=cfg.VOCAB_SIZE,       # 8192 — keeps architecture compatible
)

print(f"\nTokenizer trained.")
print(f"  Vocab size : {med_tok.get_vocab_size():,}")
print(f"  EOS token  : {med_tok.token_to_id('</s>')}")
print(f"  PAD token  : {med_tok.token_to_id('<pad>')}")

Training BPE tokenizer on : /Users/aman2394/Desktop/slm-learning/data/medical/train.txt
Target vocab size          : 8,192
Output directory           : /Users/aman2394/Desktop/slm-learning/tokenizer/medical
Training tokenizer on 1 file(s):
  /Users/aman2394/Desktop/slm-learning/data/medical/train.txt



Tokenizer saved to /Users/aman2394/Desktop/slm-learning/tokenizer/medical  (vocab size: 8192)

Tokenizer trained.
  Vocab size : 8,192
  EOS token  : 2
  PAD token  : 1


## 5 — Tokenizer Comparison

We compare the TinyStories tokenizer against the new medical tokenizer on 10 benchmark
medical terms. A good medical tokenizer should need **fewer tokens** per term — this
directly reduces sequence length and improves the model's ability to reason about
medical concepts as single units.

In [29]:
from tokenizer.preprocess import load_tokenizer
import config as cfg

ts_tok  = load_tokenizer(cfg.TOKENIZER_VOCAB,     cfg.TOKENIZER_MERGES)
med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

MEDICAL_TERMS = [
    "myocardial infarction",
    "hypertension",
    "pneumonia",
    "pharmacokinetics",
    "tachycardia",
    "stethoscope",
    "anesthesia",
    "oncology",
    "contraindication",
    "ventricular",
]

print(f"{'Term':<30} {'TS #tok':>10} {'Med #tok':>10}  Improvement")
print("-" * 65)
total_ts = total_med = 0
for term in MEDICAL_TERMS:
    ts_n  = len(ts_tok.encode(term).ids)
    med_n = len(med_tok.encode(term).ids)
    delta = ts_n - med_n
    flag  = f"↓ {delta}" if delta > 0 else ("=" if delta == 0 else f"↑ {-delta}")
    print(f"{term:<30} {ts_n:>10} {med_n:>10}  {flag}")
    total_ts  += ts_n
    total_med += med_n

print("-" * 65)
print(f"{'TOTAL':<30} {total_ts:>10} {total_med:>10}  ↓ {total_ts - total_med} tokens total")
print(f"\nAverage compression: {total_ts / total_med:.2f}x")

Term                              TS #tok   Med #tok  Improvement
-----------------------------------------------------------------
myocardial infarction                   9          5  ↓ 4
hypertension                            5          2  ↓ 3
pneumonia                               5          3  ↓ 2
pharmacokinetics                        8          4  ↓ 4
tachycardia                             6          3  ↓ 3
stethoscope                             3          4  ↑ 1
anesthesia                              4          2  ↓ 2
oncology                                4          3  ↓ 1
contraindication                        6          5  ↓ 1
ventricular                             3          1  ↓ 2
-----------------------------------------------------------------
TOTAL                                  53         32  ↓ 21 tokens total

Average compression: 1.66x


## 6 — Tokenize All Splits

`tokenize_and_save()` reads the `.txt` file line by line, encodes each line with the
given tokenizer, and writes all token IDs as a flat `uint16` numpy array to a `.npy` file.

**Why flat `uint16`?**  
- Flat array → random-access indexing: `data[i : i + block_size]` is O(1)
- `uint16` covers vocab up to 65,535 — our 8,192-token vocab fits comfortably
- Loaded with `np.memmap` — OS pages in only the slices we access during training

We temporarily patch the config paths so `tokenize_and_save` writes to the
medical paths rather than the default TinyStories paths.

**Expected time:** 20–40 minutes for ~91k documents.

In [30]:
import config as cfg
from tokenizer.preprocess import load_tokenizer, tokenize_and_save

# Load the freshly trained medical tokenizer
med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

# ── Temporarily patch config so tokenize_and_save uses medical paths ─────
# tokenize_and_save() reads cfg.{TRAIN,VAL,TEST}_TXT and writes cfg.{TRAIN,VAL,TEST}_TOKENS.
# We save the originals and restore them after the calls.
_orig = {
    "TRAIN_TXT":    cfg.TRAIN_TXT,
    "VAL_TXT":      cfg.VAL_TXT,
    "TEST_TXT":     cfg.TEST_TXT,
    "TRAIN_TOKENS": cfg.TRAIN_TOKENS,
    "VAL_TOKENS":   cfg.VAL_TOKENS,
    "TEST_TOKENS":  cfg.TEST_TOKENS,
}

cfg.TRAIN_TXT    = cfg.MED_TRAIN_TXT
cfg.VAL_TXT      = cfg.MED_VAL_TXT
cfg.TEST_TXT     = cfg.MED_TEST_TXT
cfg.TRAIN_TOKENS = cfg.MED_TRAIN_TOKENS
cfg.VAL_TOKENS   = cfg.MED_VAL_TOKENS
cfg.TEST_TOKENS  = cfg.MED_TEST_TOKENS

print("Tokenizing train split...")
tokenize_and_save("train", med_tok, batch_size=4000)

print("Tokenizing val split...")
tokenize_and_save("val", med_tok, batch_size=4000)

print("Tokenizing test split...")
tokenize_and_save("test", med_tok, batch_size=4000)

# ── Restore original config paths ─────────────────────────────────────────
for k, v in _orig.items():
    setattr(cfg, k, v)

print("\nAll splits tokenized. Config paths restored.")

Tokenizing train split...
Tokenising train: /Users/aman2394/Desktop/slm-learning/data/medical/train.txt → /Users/aman2394/Desktop/slm-learning/data/medical/train_tokens.npy


train: 91000it [00:48, 1879.75it/s]


Done: /Users/aman2394/Desktop/slm-learning/data/medical/train_tokens.npy
Tokenizing val split...
Tokenising val: /Users/aman2394/Desktop/slm-learning/data/medical/val.txt → /Users/aman2394/Desktop/slm-learning/data/medical/val_tokens.npy


val: 2000it [00:00, 211577.08it/s]


Done: /Users/aman2394/Desktop/slm-learning/data/medical/val_tokens.npy
Tokenizing test split...
Tokenising test: /Users/aman2394/Desktop/slm-learning/data/medical/test.txt → /Users/aman2394/Desktop/slm-learning/data/medical/test_tokens.npy


test: 2000it [00:00, 192629.01it/s]


Done: /Users/aman2394/Desktop/slm-learning/data/medical/test_tokens.npy

All splits tokenized. Config paths restored.


## 7 — Verify

Load each `.npy` file with `np.memmap` and print the token count.
This confirms all three files were written successfully and have reasonable sizes.

For reference: Chinchilla optimal for a 25M param model is ~500M tokens.
We expect train to be in the 400–500M range.

In [31]:
import os
import numpy as np
import config as cfg

splits = [
    ("train", cfg.MED_TRAIN_TOKENS),
    ("val",   cfg.MED_VAL_TOKENS),
    ("test",  cfg.MED_TEST_TOKENS),
]

print(f"{'Split':<10} {'Tokens':>15}  {'Millions':>10}  File")
print("-" * 75)
for name, path in splits:
    if not os.path.exists(path):
        print(f"  {name:<10} FILE NOT FOUND: {path}")
        continue
    arr = np.memmap(path, dtype=np.uint16, mode="r")
    print(f"  {name:<10} {len(arr):>15,}  {len(arr)/1e6:>10.1f}M  {path}")

print()
print("Expected: train >> val ≈ test (by the configured size ratios).")
print("Chinchilla target for 25M params: ~500M tokens.")

Split               Tokens    Millions  File
---------------------------------------------------------------------------
  train          127,214,633       127.2M  /Users/aman2394/Desktop/slm-learning/data/medical/train_tokens.npy
  val              1,860,235         1.9M  /Users/aman2394/Desktop/slm-learning/data/medical/val_tokens.npy
  test             1,646,342         1.6M  /Users/aman2394/Desktop/slm-learning/data/medical/test_tokens.npy

Expected: train >> val ≈ test (by the configured size ratios).
Chinchilla target for 25M params: ~500M tokens.


## Next Steps

The medical corpus is ready. Next:

- **Notebook 11** — Medical Pretraining: initialize a fresh GPT from random weights and
  pretrain on the medical corpus for 20k steps.
- The tokenizer files in `cfg.MED_TOKENIZER_DIR` will be used by every subsequent notebook.

To push the tokenizer to HuggingFace Hub now (optional):
```python
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path=cfg.MED_TOKENIZER_DIR,
    repo_id=cfg.MED_HF_TOKENIZER_REPO,
    repo_type="model",
)
```